In [1]:
#importing all necessary imports
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier

import sklearn.model_selection as skmodel
import sklearn.neighbors as skng
import sklearn.metrics as skmet

import pandas as pd
import joblib

In [2]:
#importing the train dataset 
df = pd.read_csv("datasets/train.csv").drop(columns=['User_ID'])
df.head(10)

,Age,Gender,Platform,Daily_Usage_Time (minutes),Posts_Per_Day,Likes_Received_Per_Day,Comments_Received_Per_Day,Messages_Sent_Per_Day,Dominant_Emotion
0,25,Female,Instagram,120.0,3.0,45.0,10.0,12.0,Happiness
1,30,Male,Twitter,90.0,5.0,20.0,25.0,30.0,Anger
2,22,Male,Facebook,60.0,2.0,15.0,5.0,20.0,Neutral
3,28,Female,Instagram,200.0,8.0,100.0,30.0,50.0,Anxiety
4,33,Male,LinkedIn,45.0,1.0,5.0,2.0,10.0,Boredom
5,21,Male,Instagram,150.0,4.0,60.0,15.0,25.0,Happiness
6,27,Female,Twitter,85.0,3.0,30.0,10.0,18.0,Anger
7,24,Male,Facebook,110.0,6.0,25.0,12.0,22.0,Sadness
8,29,Female,LinkedIn,55.0,2.0,10.0,3.0,8.0,Neutral
9,31,Male,Instagram,170.0,5.0,80.0,20.0,35.0,Happiness


In [3]:
#importing the test dataset 
df_test = pd.read_csv("datasets/test.csv").drop(columns=['User_ID'])
df_test.head(10)

,Age,Gender,Platform,Daily_Usage_Time (minutes),Posts_Per_Day,Likes_Received_Per_Day,Comments_Received_Per_Day,Messages_Sent_Per_Day,Dominant_Emotion
0,27,Female,Snapchat,120,4,40,18,22,Neutral
1,21,Male,Snapchat,60,1,18,7,12,Neutral
2,28,Male,Snapchat,115,3,38,18,27,Anxiety
3,27,Male,Telegram,105,3,48,20,28,Anxiety
4,21,Female,Facebook,55,3,17,7,12,Neutral
5,25,Female,Instagram,160,6,85,26,30,Happiness
6,25,Female,Instagram,160,6,85,26,30,Happiness
7,21,Female,Facebook,55,3,17,7,12,Neutral
8,24,Female,Snapchat,85,3,33,20,18,Happiness
9,33,Male,LinkedIn,45,1,10,5,12,Boredom


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1001 entries, 0 to 1000
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Age                         1001 non-null   object 
 1   Gender                      1000 non-null   object 
 2   Platform                    1000 non-null   object 
 3   Daily_Usage_Time (minutes)  1000 non-null   float64
 4   Posts_Per_Day               1000 non-null   float64
 5   Likes_Received_Per_Day      1000 non-null   float64
 6   Comments_Received_Per_Day   1000 non-null   float64
 7   Messages_Sent_Per_Day       1000 non-null   float64
 8   Dominant_Emotion            1000 non-null   object 
dtypes: float64(5), object(4)
memory usage: 70.5+ KB


In [5]:
#checking on the Age column
df["Age"].unique()

array(['25', '30', '22', '28', '33', '21', '27', '24', '29', '31', '23',
       '26', '34', '35', '32', 'Male', 'Female',
       ' işte mevcut veri kümesini 1000 satıra tamamlıyorum:'],
      dtype=object)

In [6]:
#dropping the faulty row
temp = " işte mevcut veri kümesini 1000 satıra tamamlıyorum:"

df = df[df["Age"] != temp]
df["Age"].value_counts()

Age
27        92
28        92
29        90
22        74
26        66
24        64
25        64
31        62
21        56
33        56
30        48
23        48
Male      40
35        38
32        38
34        36
Female    36
Name: count, dtype: int64

In [7]:
#reswap values
def swap_incorrect_values(row):
    if isinstance(row['Age'], str) and isinstance(row['Gender'], int):
        row['Age'], row['Gender'] = row['Gender'], row['Age']
    return row


df.apply(swap_incorrect_values, axis=1)
df.sample(10)  

,Age,Gender,Platform,Daily_Usage_Time (minutes),Posts_Per_Day,Likes_Received_Per_Day,Comments_Received_Per_Day,Messages_Sent_Per_Day,Dominant_Emotion
979,33,Male,Facebook,65.0,1.0,13.0,7.0,14.0,Anxiety
795,26,Male,Instagram,150.0,5.0,70.0,25.0,30.0,Anxiety
936,24,Female,LinkedIn,55.0,1.0,11.0,6.0,11.0,Boredom
788,Female,21,Snapchat,60.0,1.0,18.0,7.0,12.0,Neutral
888,21,Female,Snapchat,60.0,1.0,18.0,7.0,12.0,Neutral
551,24,Female,Twitter,75.0,3.0,33.0,18.0,19.0,Anger
706,21,Male,Instagram,150.0,5.0,65.0,25.0,30.0,Happiness
104,33,Male,LinkedIn,45.0,1.0,10.0,5.0,12.0,Boredom
434,29,Male,Facebook,65.0,2.0,27.0,11.0,22.0,Anxiety
211,26,Male,Facebook,95.0,3.0,30.0,14.0,18.0,Sadness


In [8]:
#the data set is balanced
df['Dominant_Emotion'].value_counts()

Dominant_Emotion
Happiness    200
Neutral      200
Anxiety      170
Sadness      160
Boredom      140
Anger        130
Name: count, dtype: int64

In [9]:
#ensuring that the data is in the correct range
expected_genders = ['Male', 'Female']
expected_platforms = ['Instagram', 'Twitter', 'Facebook', 'LinkedIn', 'Snapchat', 'Telegram']

df = df[df['Gender'].isin(expected_genders)]
df_test = df_test[df_test['Gender'].isin(expected_genders)]

df = df[df['Platform'].isin(expected_platforms)]
df_test = df_test[df_test['Platform'].isin(expected_platforms)]

In [10]:
#converting numerical columns to integer
columns = ['Age','Daily_Usage_Time (minutes)','Posts_Per_Day','Likes_Received_Per_Day','Comments_Received_Per_Day','Messages_Sent_Per_Day']

for col in columns:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

In [11]:
#combining train and test data set for the next step
combined_df = pd.concat([df, df_test])

In [12]:
#encoding all the categorical data
label_encoder_platform = LabelEncoder()
label_encoder_gender = LabelEncoder()
label_encoder_emotion = LabelEncoder()

combined_df["Platform"] = label_encoder_platform.fit_transform(combined_df["Platform"])

combined_df["Gender"] = label_encoder_gender.fit_transform(combined_df["Gender"])

combined_df["Dominant_Emotion"] = label_encoder_emotion.fit_transform(combined_df["Dominant_Emotion"])

In [13]:
#separating data back to train and test datasets
df["Platform"] = combined_df["Platform"].iloc[:len(df)]

df["Gender"] = combined_df["Gender"].iloc[:len(df)]

df["Dominant_Emotion"] = combined_df["Dominant_Emotion"].iloc[:len(df)]

In [14]:
df_test["Platform"] = combined_df["Platform"].iloc[len(df):]

df_test["Gender"] = combined_df["Gender"].iloc[len(df):]

df_test["Dominant_Emotion"] = combined_df["Dominant_Emotion"].iloc[len(df):]

In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 856 entries, 0 to 1000
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   Age                         856 non-null    int32
 1   Gender                      856 non-null    int32
 2   Platform                    856 non-null    int32
 3   Daily_Usage_Time (minutes)  856 non-null    int32
 4   Posts_Per_Day               856 non-null    int32
 5   Likes_Received_Per_Day      856 non-null    int32
 6   Comments_Received_Per_Day   856 non-null    int32
 7   Messages_Sent_Per_Day       856 non-null    int32
 8   Dominant_Emotion            856 non-null    int32
dtypes: int32(9)
memory usage: 36.8 KB


In [16]:
df.sample(10)

,Age,Gender,Platform,Daily_Usage_Time (minutes),Posts_Per_Day,Likes_Received_Per_Day,Comments_Received_Per_Day,Messages_Sent_Per_Day,Dominant_Emotion
468,27,1,4,105,3,48,20,28,1
552,29,0,0,85,2,27,9,20,4
381,24,0,3,85,3,33,20,18,3
723,29,1,2,60,2,17,8,15,2
342,29,1,5,95,4,50,22,22,0
232,23,0,5,70,3,30,14,18,4
888,21,0,3,60,1,18,7,12,4
521,24,0,5,105,4,35,20,23,0
832,28,1,1,145,7,80,30,35,3
325,31,1,0,80,2,20,10,20,4


In [17]:
df_test.sample(10)

,Age,Gender,Platform,Daily_Usage_Time (minutes),Posts_Per_Day,Likes_Received_Per_Day,Comments_Received_Per_Day,Messages_Sent_Per_Day,Dominant_Emotion
62,29,1,0,60,1,12,4,12,2
6,25,0,1,160,6,85,26,30,3
90,23,0,2,50,1,12,6,12,4
26,27,1,0,60,1,12,5,15,2
46,34,1,2,65,1,14,6,15,2
44,30,1,5,70,2,30,12,20,1
7,21,0,0,55,3,17,7,12,4
85,28,0,1,200,8,110,40,45,1
69,30,1,1,175,7,100,33,38,3
98,24,1,0,110,6,35,23,25,5


In [18]:
x = df[["Age", "Platform", "Comments_Received_Per_Day", "Likes_Received_Per_Day", "Gender", "Messages_Sent_Per_Day", "Posts_Per_Day", "Daily_Usage_Time (minutes)"]]
y = df["Dominant_Emotion"]

x_test2 = df_test[["Age", "Platform", "Comments_Received_Per_Day", "Likes_Received_Per_Day", "Gender", "Messages_Sent_Per_Day", "Posts_Per_Day", "Daily_Usage_Time (minutes)"]]

In [19]:
x.sample(10)

,Age,Platform,Comments_Received_Per_Day,Likes_Received_Per_Day,Gender,Messages_Sent_Per_Day,Posts_Per_Day,Daily_Usage_Time (minutes)
157,29,3,7,18,0,18,2,50
386,31,4,20,45,1,28,3,85
101,30,5,20,30,1,22,4,90
2,22,0,5,15,1,20,2,60
716,21,0,4,10,1,12,1,40
148,32,1,28,80,1,31,5,130
152,29,0,9,27,0,20,2,85
430,34,2,5,17,0,14,2,65
533,21,1,22,95,0,28,6,130
999,28,4,8,18,0,18,2,60


In [20]:
x_test2.sample(10)

,Age,Platform,Comments_Received_Per_Day,Likes_Received_Per_Day,Gender,Messages_Sent_Per_Day,Posts_Per_Day,Daily_Usage_Time (minutes)
64,23,5,20,45,0,25,4,75
53,28,5,22,48,1,25,4,90
70,35,5,8,13,1,10,1,70
23,25,4,13,30,1,25,4,80
99,24,4,16,37,1,22,3,75
2,28,3,18,38,1,27,3,115
22,22,0,6,14,0,10,1,70
60,27,2,2,9,1,11,1,55
62,29,0,4,12,1,12,1,60
16,31,4,10,20,1,12,2,50


In [21]:
y.sample(10)

122    2
346    5
137    0
232    4
938    0
26     1
499    4
572    4
924    3
19     3
Name: Dominant_Emotion, dtype: int32

In [22]:
#training a Decision Tree Classifier with kfolds
clf = DecisionTreeClassifier(random_state=42)
k_folds = KFold(n_splits = 5)
scores = cross_val_score(clf, x, y, cv = k_folds)

print("Cross Validation Scores: ", scores)
print("Average CV Score: ", scores.mean())
print("Number of CV Scores used in Average: ", len(scores))

Cross Validation Scores:  [0.8255814 1.        1.        1.        1.       ]
Average CV Score:  0.9651162790697676
Number of CV Scores used in Average:  5


In [23]:
#training a Decision Tree Classifier
clf2 = DecisionTreeClassifier(random_state=42).fit(x,y)
res = clf2.predict(x_test2)

print(skmet.classification_report (df_test["Dominant_Emotion"] , res ))

              precision    recall  f1-score   support

           0       0.86      1.00      0.92         6
           1       1.00      0.90      0.95        21
           2       1.00      0.88      0.93        16
           3       0.93      1.00      0.96        13
           4       0.93      1.00      0.96        27
           5       0.86      0.86      0.86        14

    accuracy                           0.94        97
   macro avg       0.93      0.94      0.93        97
weighted avg       0.94      0.94      0.94        97



In [24]:
#training a Random Forest Classifier
classifier = RandomForestClassifier(n_estimators = 100).fit(x,y)
y_predict= classifier.predict(x_test2)

print(skmet.classification_report (df_test["Dominant_Emotion"] , y_predict))
print(accuracy_score(df_test["Dominant_Emotion"] , y_predict))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00         6
           1       1.00      0.95      0.98        21
           2       1.00      0.94      0.97        16
           3       0.93      1.00      0.96        13
           4       1.00      1.00      1.00        27
           5       0.93      1.00      0.97        14

    accuracy                           0.98        97
   macro avg       0.98      0.98      0.98        97
weighted avg       0.98      0.98      0.98        97

0.979381443298969


In [25]:
#training  a KNN
classifier2 = skng.KNeighborsClassifier(n_neighbors=5).fit(x,y)
y_predict2= classifier2.predict(x_test2)

print(skmet.classification_report (df_test["Dominant_Emotion"] , y_predict2))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00         6
           1       0.95      0.95      0.95        21
           2       0.88      0.94      0.91        16
           3       0.92      0.92      0.92        13
           4       1.00      0.93      0.96        27
           5       0.87      0.93      0.90        14

    accuracy                           0.94        97
   macro avg       0.94      0.94      0.94        97
weighted avg       0.94      0.94      0.94        97



In [26]:
#Saving the encoders and the wanted model
joblib.dump(classifier, 'model/classifier.joblib')
joblib.dump(label_encoder_platform, 'encoders/label_encoder_platform.joblib')
joblib.dump(label_encoder_gender, 'encoders/label_encoder_gender.joblib')
joblib.dump(label_encoder_emotion, 'encoders/label_encoder_emotion.joblib')

['encoders/label_encoder_emotion.joblib']